# 5. Pemodelan dan Evaluasi

Kedua model dilatih menggunakan sembilan fitur spektral yang telah diagregasi pada tingkat poligon dan distandarkan. Poligon Laut `Laut_0000` sampai `Laut_0018` dikecualikan, piksel non-Laut di luar batas Jawa Timur disaring, dan titik rata-rata poligon diperiksa kembali seperti pada dashboard Streamlit. Dengan dataset dan batas wilayah proyek saat ini, penyaringan menghasilkan **278 poligon**; kode menghitung jumlah tersebut ulang saat dijalankan.

## Gaussian Naive Bayes

Model dibuat menggunakan `GaussianNB`. Algoritma ini menerapkan Teorema Bayes dan memodelkan distribusi fitur tiap kelas sebagai distribusi Gaussian. Prediksi pada data uji dibandingkan dengan label sebenarnya menggunakan akurasi dan *classification report*.

Output akurasi dan laporan klasifikasi lengkap akan muncul dari sel pemodelan di bagian akhir halaman.

## K-Nearest Neighbors

Model KNN menggunakan `KNeighborsClassifier` dengan `n_neighbors=5`. Kelas suatu observasi ditentukan dari kelas mayoritas di antara lima tetangga terdekat pada ruang fitur yang telah distandarkan.

Output akurasi dan laporan klasifikasi lengkap akan muncul dari sel pemodelan di bagian akhir halaman.

## Perbandingan

Metrik dihitung ulang dari dataset setelah penyaringan pada saat sel dijalankan. Hasil per kelas, termasuk *precision*, *recall*, dan *F1-score*, perlu ditafsirkan berdasarkan output terbaru dan tidak disimpulkan dari output notebook lama.

Nilai tersebut merupakan hasil satu pembagian data dengan `random_state=42`. Pengujian dengan validasi silang atau sampel tambahan dapat memberikan gambaran kestabilan kinerja yang lebih luas.

## Menjalankan Pemodelan

Sel berikut menyiapkan data dengan proses yang sama pada bagian preprocessing, melatih kedua model, lalu menampilkan akurasi dan *classification report* setiap model. Dengan demikian, metrik di halaman ini dihitung ulang dari CSV ketika sel dijalankan.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import shape
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

dataset_path = next(
    (
        candidate
        for folder in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            folder / "piksel_s2_JTM.csv",
            folder / "Tugas" / "UTS" / "piksel_s2_JTM.csv",
        )
        if candidate.is_file()
    ),
    None,
)
if dataset_path is None:
    raise FileNotFoundError("Dataset piksel_s2_JTM.csv tidak ditemukan.")

df = pd.read_csv(dataset_path)
boundary_path = dataset_path.with_name("batas_jawa_timur.geojson")
if not boundary_path.is_file():
    raise FileNotFoundError(f"Batas Jawa Timur tidak ditemukan: {boundary_path}")
with boundary_path.open(encoding="utf-8") as file:
    boundary_feature = json.load(file)
if boundary_feature.get("properties", {}).get("adm1_pcode") != "ID35":
    raise ValueError("Berkas geometri bukan batas Provinsi Jawa Timur.")
boundary = shape(boundary_feature["geometry"])
if boundary.is_empty or not boundary.is_valid:
    raise ValueError("Geometri batas Provinsi Jawa Timur tidak valid.")
poligon_laut_dikecualikan = {f"Laut_{index:04d}" for index in range(19)}
coordinates = df[["lon", "lat"]].to_numpy(dtype=np.float64)
finite_coordinates = np.isfinite(coordinates).all(axis=1)
inside_boundary = np.zeros(len(df), dtype=bool)
inside_boundary[finite_coordinates] = shapely.contains_xy(
    boundary,
    coordinates[finite_coordinates, 0],
    coordinates[finite_coordinates, 1],
)
keep_laut = df["label_teks"].eq("Laut").to_numpy()
keep_polygon = ~df["poligon_id"].isin(poligon_laut_dikecualikan).to_numpy()
df = df.loc[finite_coordinates & keep_polygon & (inside_boundary | keep_laut)].copy()
fitur = ["B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI", "MNDWI", "NDBI"]
kolom_agregasi = ["poligon_id", "label", "label_teks", "lon", "lat", *fitur]
df_bersih = df[kolom_agregasi].replace([np.inf, -np.inf], np.nan).dropna().copy()
poligon = (
    df_bersih.groupby("poligon_id", as_index=False)
    .agg(
        label=("label", "first"),
        label_teks=("label_teks", "first"),
        lon=("lon", "mean"),
        lat=("lat", "mean"),
        **{nama_fitur: (nama_fitur, "mean") for nama_fitur in fitur},
    )
)
coordinates = poligon[["lon", "lat"]].to_numpy(dtype=np.float64)
finite_coordinates = np.isfinite(coordinates).all(axis=1)
inside_boundary = np.zeros(len(poligon), dtype=bool)
inside_boundary[finite_coordinates] = shapely.contains_xy(
    boundary,
    coordinates[finite_coordinates, 0],
    coordinates[finite_coordinates, 1],
)
keep_laut = poligon["label_teks"].eq("Laut").to_numpy()
poligon = poligon.loc[
    finite_coordinates & (inside_boundary | keep_laut)
].copy()

X_train, X_test, y_train, y_test = train_test_split(
    poligon[fitur],
    poligon["label_teks"],
    test_size=0.3,
    random_state=42,
    stratify=poligon["label_teks"],
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Gaussian Naive Bayes": GaussianNB(),
    "K-Nearest Neighbors (k=5)": KNeighborsClassifier(n_neighbors=5),
}
for nama, model in models.items():
    model.fit(X_train_scaled, y_train)
    prediksi = model.predict(X_test_scaled)
    print(f"=== {nama} ===")
    print(f"Akurasi: {accuracy_score(y_test, prediksi) * 100:.2f}%")
    print(classification_report(y_test, prediksi, zero_division=0))

=== Gaussian Naive Bayes ===
Akurasi: 71.43%
              precision    recall  f1-score   support

    Bangunan       0.76      0.87      0.81        15
       Danau       0.83      0.67      0.74        15
       Hutan       0.71      0.80      0.75        15
        Laut       0.80      0.89      0.84         9
    Mangrove       0.65      0.73      0.69        15
       Sawah       0.55      0.40      0.46        15

    accuracy                           0.71        84
   macro avg       0.72      0.73      0.72        84
weighted avg       0.71      0.71      0.71        84

=== K-Nearest Neighbors (k=5) ===
Akurasi: 73.81%
              precision    recall  f1-score   support

    Bangunan       0.71      0.80      0.75        15
       Danau       0.91      0.67      0.77        15
       Hutan       0.64      0.93      0.76        15
        Laut       0.90      1.00      0.95         9
    Mangrove       0.85      0.73      0.79        15
       Sawah       0.55      0.40    